# Table 2 — Global Moran's I (Multiple Specifications)

**Script file:** `Paper1_Table2_morans_i.ipynb`

**หมายเหตุ:** เทียบผลลัพธ์กับ Table 2 ในต้นฉบับก่อนใช้แทน หากคลาดเคลื่อนให้ยึดต้นฉบับเป็นหลัก

**สิ่งที่ต้องเตรียมก่อนรัน:** ไฟล์ `บันทึกการตรวจวัดปริมาณฝุ่น.xlsx` (ข้อมูลภาคสนามต้นฉบับ 32 จุด)


In [ ]:
# SCRIPT: Paper1_Table2_morans_i.ipynb
# SECTION: 0 - Setup
!pip install requests openpyxl esda libpysal -q

In [ ]:
import re, time, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import openpyxl
from datetime import datetime

plt.rcParams['figure.dpi'] = 120

## 1. อัปโหลดไฟล์ข้อมูลภาคสนามต้นฉบับ

In [ ]:
# SCRIPT: Paper1_Table2_morans_i.ipynb
# SECTION: 1 - Load raw field data
from google.colab import files
uploaded = files.upload()  # เลือกไฟล์ "บันทึกการตรวจวัดปริมาณฝุ่น.xlsx"
RAW_XLSX = list(uploaded.keys())[0]
print("อัปโหลดไฟล์แล้ว:", RAW_XLSX)

In [ ]:
# SCRIPT: Paper1_Table2_morans_i.ipynb
# SECTION: 1 - Parse raw field data (name, address, link, PM2.5, TPM, date)
wb = openpyxl.load_workbook(RAW_XLSX, data_only=True)
ws = wb['Sheet1']

records = {}
order = []
for row in ws.iter_rows(min_row=3, max_row=1001, values_only=True):
    name = row[0]
    if not name:
        continue
    if name not in records:
        records[name] = {
            'name': name, 'url': row[1], 'address': row[2],
            'pm25': row[6], 'tpm': row[9], 'date': str(row[13]),
        }
        order.append(name)

sites_raw = [records[n] for n in order]
print(f"จำนวนจุดตรวจวัดทั้งหมด: {len(sites_raw)}")
assert len(sites_raw) == 32, "คาดว่าจะมี 32 จุด - ตรวจสอบไฟล์ต้นฉบับถ้าไม่ตรง" 

## 2. Resolve พิกัด GPS จากลิงก์ Google Maps

ต้องรันบน Colab (ต้องมีอินเทอร์เน็ตเปิด) เพราะ short link (`goo.gl/maps/...`) ต้องไล่ตาม redirect

In [ ]:
# SCRIPT: Paper1_Table2_morans_i.ipynb
# SECTION: 2 - Resolve GPS coordinates from Google Maps short links
PATTERNS = [
    re.compile(r"!3d(-?\d+\.\d+)!4d(-?\d+\.\d+)"),
    re.compile(r"@(-?\d+\.\d+),(-?\d+\.\d+)"),
    re.compile(r"[?&]q=(-?\d+\.\d+),(-?\d+\.\d+)"),
    re.compile(r"ll=(-?\d+\.\d+),(-?\d+\.\d+)"),
]

def extract_latlon(url_or_text):
    for pat in PATTERNS:
        m = pat.search(url_or_text)
        if m:
            return float(m.group(1)), float(m.group(2))
    return None, None

def resolve_link(short_url, timeout=15):
    try:
        r = requests.get(short_url, allow_redirects=True, timeout=timeout,
                          headers={'User-Agent': 'Mozilla/5.0'})
        lat, lon = extract_latlon(r.url)
        if lat is None:
            lat, lon = extract_latlon(r.text[:5000])
        return lat, lon
    except Exception:
        return None, None

for i, s in enumerate(sites_raw, 1):
    lat, lon = resolve_link(s['url'])
    s['lat'], s['lon'] = lat, lon
    print(f"[{i:02d}/32] {s['name']}: {'OK' if lat else 'FAILED - resolve manually'}")
    time.sleep(0.4)

missing = [s['name'] for s in sites_raw if s['lat'] is None]
if missing:
    print("\nจุดที่ resolve ไม่สำเร็จ (ต้องเติมพิกัดด้วยมือ):", missing)
else:
    print("\nพิกัดครบทั้ง 32 จุด")

In [ ]:
# SCRIPT: Paper1_Table2_morans_i.ipynb
# SECTION: 2 - Manual coordinate override / field verification note
# sites_raw[<index>]['lat'], sites_raw[<index>]['lon'] = <lat>, <lon>

# ยืนยันภาคสนามแล้ว: Site 13 (Tab Kwang) อยู่ติดถนนมิตรภาพ ตรงข้าม TPI Polene
# พิกัดยืนยัน: 14 38 09.6 N, 101 06 55.3 E (คลาดเคลื่อนจากพิกัด resolve <300 m)

df = pd.DataFrame(sites_raw)
df['is_purposive'] = df['name'].str.contains('survey')
df[['name', 'address', 'lat', 'lon', 'pm25', 'tpm', 'date', 'is_purposive']]

## 3. สร้าง Table 2

In [ ]:
# SCRIPT: Paper1_Table2_morans_i.ipynb
# SECTION: helper - build k-NN and distance-band spatial weights + Moran's I
import libpysal
from esda.moran import Moran

coords = np.column_stack([df['lon'].values, df['lat'].values])
pm25 = df['pm25'].values
logpm25 = np.log(pm25)

w_knn = libpysal.weights.KNN.from_array(coords, k=5); w_knn.transform = 'r'
w_dist = libpysal.weights.DistanceBand.from_array(coords, threshold=0.5, binary=True); w_dist.transform = 'r'

def moran_row(vals, w, dataset_label, weights_label, var_label):
    m = Moran(vals, w, permutations=999)
    return {'Dataset': dataset_label, 'Weights': weights_label, 'Variable': var_label,
             'I': round(m.I, 4), 'z': round(m.z_sim, 3), 'p': round(m.p_sim, 3)}

In [ ]:
# SCRIPT: Paper1_Table2_morans_i.ipynb
# SECTION: 3 - TABLE 2: Global Moran's I under alternative weights/transform/outlier treatment
table2_rows = []
table2_rows.append(moran_row(pm25,    w_knn,  'Full (n=32)', 'k-NN (k=5)',      'Raw PM2.5'))
table2_rows.append(moran_row(logpm25, w_knn,  'Full (n=32)', 'k-NN (k=5)',      'Log PM2.5'))
table2_rows.append(moran_row(pm25,    w_dist, 'Full (n=32)', 'Distance-band',   'Raw PM2.5'))
table2_rows.append(moran_row(logpm25, w_dist, 'Full (n=32)', 'Distance-band',   'Log PM2.5'))

mask = df['name'] != 'จุดตรวจวัดอากาศ-13'
coords_ex = coords[mask.values]
pm25_ex = pm25[mask.values]; logpm25_ex = logpm25[mask.values]
w_knn_ex = libpysal.weights.KNN.from_array(coords_ex, k=5); w_knn_ex.transform = 'r'
w_dist_ex = libpysal.weights.DistanceBand.from_array(coords_ex, threshold=0.5, binary=True); w_dist_ex.transform = 'r'
table2_rows.append(moran_row(pm25_ex,    w_knn_ex,  'Excl. Tab Kwang (n=31)', 'k-NN (k=5)',    'Raw PM2.5'))
table2_rows.append(moran_row(logpm25_ex, w_knn_ex,  'Excl. Tab Kwang (n=31)', 'k-NN (k=5)',    'Log PM2.5'))
table2_rows.append(moran_row(pm25_ex,    w_dist_ex, 'Excl. Tab Kwang (n=31)', 'Distance-band', 'Raw PM2.5'))
table2_rows.append(moran_row(logpm25_ex, w_dist_ex, 'Excl. Tab Kwang (n=31)', 'Distance-band', 'Log PM2.5'))

table2 = pd.DataFrame(table2_rows)
print(table2.to_string(index=False))
table2.to_excel('Table2_morans_i.xlsx', index=False)
print("\nNOTE: compare against manuscript Table 2 before relying on these values.")

## ดาวน์โหลดผลลัพธ์

In [ ]:
# SCRIPT: Paper1_Table2_morans_i.ipynb
# SECTION: Download output
from google.colab import files as gfiles
for f in ['Table2_morans_i.xlsx']:
    gfiles.download(f)